# Figure 3 · Beyond the training horizon (Dark Key-to-Door)

The paper's long-run result, and the figure it stands on: frozen 8M endpoints of the memory carriers (RSM-R and RSM-O, the full-history Transformer, the GRU, published Memo; Memo with fixed segments in the appendix set; RSM no carry has nothing to retain and stays in Figures 2 and 4) run on one Dark Key-to-Door task far past the 500-call budget they were trained on, on the longest horizon every cell has finished (4,000 calls when complete). Three panels: (a) completed doors per 500-call window against charged calls, the trained budget marked and the extension shaded, seeds as thin lines; (b) cumulative completed doors over the same task; (c) the last window's door rate against the average live inference state an actor holds over the task (log x, one point per cell with its seeds behind it, no connecting frontier). The paired H4a/H4b readings are printed under the figure.

Appendix A5 (`figureA5_…`, built below from the same records) keeps every benchmark on one plot: the same Key-to-Door row beside CountRecall retention within the trained stream, and MazeRunner goal fraction against maze size under the area-scaled budget, with a second row of retained quality against the **peak** live inference state an actor holds within the task (the memory a deployment has to provide; the mean over the task stays in the saved table).

Extrapolation on Key-to-Door and the larger mazes, never matched train/test; the CountRecall panel is retention inside the trained horizon and is never called extrapolation. State is one accounting for every cell: Memo its recorded per-slot schedule, the full history one cache slot per record, every other cell its measured fixed buffer. A cell without complete panels on a roster is named in the legend as pending; nothing about it is written on a chart. Rules are stated here; the estimators and the plotting code are in this notebook, the shared plumbing in [figures_common.py](figures_common.py).

In [ ]:
"""Locate the repository, the study roots and the output folders.

Environment overrides, for a copy of the records elsewhere:
  REASONED_ICRL_OUTPUTS   study roots (default <repo>/outputs)
  FIGURE_OUTPUT_ROOT      where the figure is written (default notebooks/outputs/figures);
                          its tables and notes go to the sibling data/ folder
"""
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
sys.path.insert(0, str(REPO / "notebooks"))

from figures_common import default_layout, markdown_preview  # noqa: E402

OUTPUTS = Path(os.environ.get("REASONED_ICRL_OUTPUTS", REPO / "outputs"))
FIGURES_DIR = Path(
    os.environ.get("FIGURE_OUTPUT_ROOT", REPO / "notebooks/outputs/figures")
)
DATA_DIR = FIGURES_DIR.parent / "data"
layout = default_layout(REPO, outputs=OUTPUTS)
print(f"records under {OUTPUTS}\nfigures to  {FIGURES_DIR}\ntables to   {DATA_DIR}")

In [ ]:
import json
import textwrap
from collections.abc import Sequence
from dataclasses import dataclass

import numpy as np
from figures_common import (
    ABLATION_CONDITION,
    DRAWN_CELLS,
    ENVIRONMENTS,
    FULL_WIDTH,
    INK_FAINT,
    INK_MUTED,
    METHOD_CONDITION,
    PANEL,
    PAPER_ENVIRONMENTS,
    SEEDS,
    SUMMARY_CLEARED,
    BuiltFigure,
    CellStyle,
    band,
    cell_state_rows,
    composite_legend,
    endpoint_panel,
    horizon_panel,
    legend_handles,
    load_cell,
    load_streams_cell,
    maze_budget,
    mean_live_state_bytes,
    paper_style,
    pending_handles,
    standalone,
    stratify,
    style_axes,
    write_strata,
)
from matplotlib.lines import Line2D

from reasoned_icrl.analysis.statistics import _bootstrap, _interval
from reasoned_icrl.experiments.horizon import (
    MAZE_SIZES,
    WINDOW,
    maze_goal_fraction,
    streams_matrix,
    window_matrix,
)

STREAM_COUNTS = (32, 16, 8, 4, 2)
"""The declared continued lengths of CountRecall in deck pairs (C3, appendix),
longest first (the figure's panel is the longest one every cell has on every
seed; the trained lengths are one to three pairs)."""
C2_TRAINED_PAIRS = 1
"""Pairs per training task: one (the continued pairs of C3 start after it)."""

KEYDOOR_HORIZONS = (4000, 2000, 1000)
"""The declared extended Key-to-Door budgets in charged calls, longest first
(500 is the trained budget; the figure's row is the longest complete one)."""
DELTA = 1.0
"""The declared practical margin on Key-to-Door: one door per 500-call window."""
SHORT = {
    "raw_summary": "RSM-O",
    "raw_segment": "w/o memory",
    "raw_summary_residual": "RSM-R",
    "full_context": "full history",
    "full_gru": "GRU",
    "memo": "Memo",
    "memo_fixed": "Memo fixed",
}
"""Point labels of the state panels."""
HORIZON_CELLS = tuple(c for c in DRAWN_CELLS if c.condition != "raw_segment")
"""The cells of Figure 3: every memory
carrier — the RSM family (RSM-R, RSM-O), the full history, the GRU, published
Memo and Memo with fixed segments. The memoryless control has nothing to
retain beyond the horizon and stays in Figures 2 and A1 and in the tables."""
APPENDIX_HORIZON_CELLS = HORIZON_CELLS
"""The appendix figure draws the same carriers."""
WRITE_RULE = {"raw_summary": "RSM-O", "raw_summary_residual": "RSM-R"}
"""The RSM family's names in contrast labels, by the write rule's cell."""


def rsm_name(condition):
    """``RSM-R`` / ``RSM-O`` for the family's cells, the cell's short name else."""
    return WRITE_RULE.get(condition, SHORT.get(condition, condition))


COUNT_RECALL_STRATA = (0, 1, 2, 3)
"""Segments since the queried suit was first dealt: 0 = still in the open segment, 1-3 = the summary must carry it."""


@dataclass(frozen=True)
class Figure3Options:
    seeds: Sequence[int] = SEEDS
    samples: int = 2000
    resampling_seed: int = 0
    split: str = "confirmation"
    """The only roster with horizon panels: the paper's roster."""
    cells: Sequence[CellStyle] = HORIZON_CELLS
    method: str = METHOD_CONDITION
    """The paper's RSM: the cell whose summary-cleared trace is drawn, whose
    last window the H4 readings compare, and whose panel fixes the CountRecall
    strata (the fixed C32 schedule; the overwrite cell stands in while the
    method's panel is absent)."""
    horizons: Sequence[int] = KEYDOOR_HORIZONS
    streams: Sequence[int] = STREAM_COUNTS
    sizes: Sequence[int] = MAZE_SIZES
    environments: Sequence[str] = ("keydoor",)
    """The composite's columns. The main text's Figure 3 is Dark Key-to-Door
    alone (``("keydoor",)``): doors per 500-call window, cumulative doors and
    the last window against live state. The appendix figure passes the three
    benchmarks (``PAPER_ENVIRONMENTS``), one beyond-horizon axis per benchmark
    above a row of quality against live state. Every environment's data and
    tables are computed and saved whatever is drawn."""
    name: str = "figure3_beyond_the_training_horizon"
    compute: Sequence[str] | None = None
    """Environments whose records are read (``None``: every environment, so
    the tables of every benchmark are saved whatever is drawn). A build that
    draws one environment can restrict the reads to it: the Key-to-Door
    4,000-call panels are the expensive load."""
    figsize: tuple[float, float] | None = None
    """Composite size; ``None`` picks it from the number of columns."""
    caption: bool = True
    """Print the paired readings under the composite."""
    show_cleared: bool = True
    """Draw the method's summary-cleared trace where the panel has one. The
    appendix figure leaves it out: it is the
    intervention, not a carrier, and it is drawn in Figure 2's panels and in
    the boundary-segment figure. The panel is saved in the tables either
    way."""
    state_accounting: str = "peak"
    """Which reading of live inference state the state row draws: ``"peak"``, the most an actor holds at any record of the
    task — the memory a deployment has to provide, and the reading the tables
    quote — or ``"mean"``, the average over the task's records. Both are in
    every saved state table, under one growth rule per cell."""
    panel_titles: str = "environment"
    """How a panel is titled and how much its labels spell out.
    ``"environment"``: the benchmark and the axis, with full labels (a
    standalone panel).
    ``"short"``: the benchmark is named once above the figure and the panels
    carry (a), (b), (c) — the main Key-to-Door figure, whose three panels are
    one benchmark.
    ``"compact"``: the benchmark names its column once, on the top row, and
    the labels drop what the caption already says — the appendix figure, whose
    three columns leave about two inches each."""

    @property
    def compact(self) -> bool:
        return self.panel_titles == "compact"

    panel_size: tuple[float, float] = PANEL


# ---------------------------------------------------------------- estimators
# Every Key-to-Door number is a mean over the roster's (task, rollout) units and
# the three seeds, paired unit by unit across cells; the interval is the study's
# joint seed/task bootstrap, conditional on the trained seeds.


def estimate(matrix, *, samples, rng, seeds):
    """Mean of a ``[seeds, units]`` matrix, its joint bootstrap interval, the seed means."""
    value = float(matrix.mean())
    lower, upper = _interval(_bootstrap(matrix, samples=samples, rng=rng), value, 0.95)
    return {
        "estimate": value,
        "lower": lower,
        "upper": upper,
        **{f"seed_{s}": float(matrix[i].mean()) for i, s in enumerate(seeds)},
    }


def load_horizon_cube(layout, condition, horizon, contract, options, sources):
    """``[seeds, units, windows]`` completed doors per 500-call window for one
    cell at one horizon, or ``None`` when any seed lacks a complete panel."""
    units = [
        (int(task), int(rollout))
        for task in contract.roster(options.split)
        for rollout in contract.evaluation.rollout_seeds
    ]
    layers, staged = [], []
    for seed in options.seeds:
        events = horizon_panel(
            layout,
            "keydoor",
            condition,
            seed,
            options.split,
            "retained",
            horizon,
            contract=contract,
            sources=staged,
        )
        if events is None:
            return None
        layers.append(window_matrix(events, units=units, outer_length=horizon))
    sources.extend(staged)
    return np.stack(layers)


def choose_horizon(layout, contract, options, sources):
    """The longest declared horizon at which every cell has complete panels;
    otherwise the horizon holding the most cells (ties to the longer), so the
    competing methods share one panel; ``None`` when no cell has any."""
    candidates = []
    for horizon in options.horizons:
        staged = []
        cubes = {
            c.condition: load_horizon_cube(
                layout, c.condition, horizon, contract, options, staged
            )
            for c in options.cells
        }
        present = sum(cube is not None for cube in cubes.values())
        if present == len(options.cells):
            sources.extend(staged)
            return horizon, cubes
        candidates.append((present, horizon, cubes, staged))
    present, horizon, cubes, staged = max(candidates, key=lambda c: c[0])
    if present == 0:
        return None, {c.condition: None for c in options.cells}
    sources.extend(staged)
    return horizon, cubes


def window_rates(cubes, *, samples, seed, seeds):
    """Per cell and window: the rate and the running total, with intervals."""
    rng = np.random.default_rng(seed)
    rows = []
    for condition, cube in cubes.items():
        if cube is None:
            continue
        running = cube.cumsum(axis=2)
        for w in range(cube.shape[2]):
            rate = estimate(cube[:, :, w], samples=samples, rng=rng, seeds=seeds)
            total = estimate(running[:, :, w], samples=samples, rng=rng, seeds=seeds)
            rows.append(
                {
                    "condition": condition,
                    "window": w + 1,
                    "first_call": w * WINDOW + 1,
                    "last_call": (w + 1) * WINDOW,
                    "centre": (w + 0.5) * WINDOW,
                    **rate,
                    **{f"cumulative_{k}": v for k, v in total.items()},
                }
            )
    return rows


def paired_contrasts(
    cubes, *, samples, seed, seeds, delta=DELTA, method=METHOD_CONDITION
):
    """H4a (the method's last window against its first and second) and the
    last-window differences against every other cell, paired per unit, with
    the reader's flags: a consistent gain (at or above delta, every seed
    positive, joint interval above zero), within delta on every seed, and no
    fall beyond delta on any seed. "RSM" names the method whichever cell it
    is; the rewrite ablation row (RSM - RSM overwrite) exists only when the
    method is the residual rewrite."""
    rng = np.random.default_rng(seed)
    rows = []

    def add(name, left, right, against):
        e = estimate(left - right, samples=samples, rng=rng, seeds=seeds)
        per = [e[f"seed_{s}"] for s in seeds]
        rows.append(
            {
                "contrast": name,
                "against": against,
                **e,
                "positive_seeds": sum(v > 0 for v in per),
                "consistent_gain": bool(
                    e["estimate"] >= delta
                    and all(v > 0 for v in per)
                    and e["lower"] > 0
                ),
                "within_delta_every_seed": all(abs(v) <= delta for v in per),
                "no_fall_every_seed": all(v >= -delta for v in per),
            }
        )

    own = cubes.get(method)
    if own is None:
        return rows
    last = own.shape[2] - 1
    add(
        f"H4a {rsm_name(method)} last window - first window",
        own[:, :, last],
        own[:, :, 0],
        "first window",
    )
    if last >= 1:
        add(
            "RSM last window - second window",
            own[:, :, last],
            own[:, :, 1],
            "second window",
        )
    family = [
        (f"H4b {rsm_name(method)} - Memo", method, "memo"),
        (f"H4b {rsm_name(method)} - full history", method, "full_context"),
        (f"{rsm_name(method)} - GRU", method, "full_gru"),
        (f"{rsm_name(method)} - Memo fixed", method, "memo_fixed"),
        (f"{rsm_name(method)} - no carry", method, "raw_segment"),
        ("Memo fixed - Memo", "memo_fixed", "memo"),
    ]
    if method != ABLATION_CONDITION:
        family.append(
            (
                f"{rsm_name(method)} - {rsm_name(ABLATION_CONDITION)}",
                method,
                ABLATION_CONDITION,
            )
        )
    for name, left, right in family:
        if cubes.get(left) is not None and cubes.get(right) is not None:
            add(name, cubes[left][:, :, last], cubes[right][:, :, last], "last window")
    return rows


def prefix_status(layout, condition, seed, horizon, split):
    """The adapter's acceptance check of one panel: its 500-call prefix must
    reproduce the trained-length panel (``missing`` when never checked)."""
    run = layout.run_directory("keydoor", condition, seed)
    path = run / "eval" / f"{split}-retained-endpoint-h{horizon}" / "prefix_check.json"
    if not path.is_file():
        return {"status": "missing", "base_panel": None}
    record = json.loads(path.read_text())
    return {"status": record.get("status"), "base_panel": record.get("base_panel")}


# ------------------------------------------------------------------- state
# Live inference state per actor comes from figures_common under one growth
# rule per cell (Memo its recorded per-slot schedule, the full history one
# cache slot per record, every other cell its measured fixed state). The
# figure's state row draws the mean over the task's records; the peak within the task and the allocation stay in the saved table.


def state_bytes_at(systems, condition, horizon, *, segment_length, summary_tokens):
    """Key-to-Door mean live state over a task of ``horizon`` charged calls
    (``horizon + 1`` records); kept as the notebook's name for the shared rule."""
    return mean_live_state_bytes(
        systems,
        condition,
        horizon + 1,
        segment_length=segment_length,
        summary_tokens=summary_tokens,
    )


def cell_states(layout, contract, cubes, horizon, options, sources):
    """One state-bytes row per present Key-to-Door cell at the horizon."""
    if horizon is None:
        return []
    present = [c.condition for c in options.cells if cubes.get(c.condition) is not None]
    return cell_state_rows(
        layout,
        "keydoor",
        present,
        horizon,
        contract=contract,
        seeds=options.seeds,
        sources=sources,
        accounting=options.state_accounting,
    )


# -------------------------------------------------------------- CountRecall
# Retention inside the trained stream: accuracy by the number of C32 segments
# since the queried suit was first dealt, every cell on the same queries.


def countrecall_retention(layout, contract, options, sources):
    """Per cell and stratum: exact accuracy over the stratum's queries, one value
    per stream, with the joint seed/task interval and the opportunity count;
    the method's summary-cleared panel on the same strata; the whole stream
    accuracy and live state per cell for the state panel."""
    native = int(contract.environment.horizon)
    rng = np.random.default_rng(options.resampling_seed)
    panels, staged = {}, []
    for cell in options.cells:
        events = load_cell(
            layout,
            "countrecall",
            cell.condition,
            options.split,
            "retained",
            contract=contract,
            seeds=options.seeds,
            sources=staged,
        )
        if events is not None:
            panels[cell.condition] = events
    data = {
        "present": [c for c in options.cells if c.condition in panels],
        "missing": [c.condition for c in options.cells if c.condition not in panels],
        # The strata depend on the task and the fixed C32 schedule, so the
        # overwrite cell's panel fixes them while the method's is absent.
        "reference": next(
            (
                c
                for c in dict.fromkeys(
                    (options.method, METHOD_CONDITION, ABLATION_CONDITION)
                )
                if c in panels
            ),
            None,
        ),
        "rows": [],
        "cleared": [],
        "endpoints": [],
        "states": [],
        "horizon": native,
    }
    if data["reference"] is None:
        return data
    sources.extend(staged)
    strata = write_strata(panels[data["reference"]])

    def rows_for(condition, events, history):
        out = []
        grouped = stratify(events, strata)
        for stratum in COUNT_RECALL_STRATA:
            queries = grouped.get(stratum, [])
            if not queries:
                continue
            per_task = {}
            for e in queries:
                per_task.setdefault(
                    (e.training_seed, e.task_id, e.rollout_seed), []
                ).append(e.numerator)
            matrix, seeds = _matrix(per_task, options.seeds)
            out.append(
                {
                    "condition": condition,
                    "history": history,
                    "stratum": stratum,
                    **estimate(matrix, samples=options.samples, rng=rng, seeds=seeds),
                    "queries": len(queries),
                    "queries_per_seed": len(queries) // max(len(seeds), 1),
                }
            )
        return out

    for cell in data["present"]:
        data["rows"].extend(
            rows_for(cell.condition, panels[cell.condition], "retained")
        )
        per_task = {}
        for e in panels[cell.condition]:
            if e.kind == "query":
                per_task.setdefault(
                    (e.training_seed, e.task_id, e.rollout_seed), []
                ).append(e.numerator)
        matrix, seeds = _matrix(per_task, options.seeds)
        data["endpoints"].append(
            {
                "condition": cell.condition,
                **estimate(matrix, samples=options.samples, rng=rng, seeds=seeds),
            }
        )
    cleared = (
        load_cell(
            layout,
            "countrecall",
            options.method,
            options.split,
            "summary-cleared",
            contract=contract,
            seeds=options.seeds,
            sources=sources,
        )
        if options.method in panels
        else None
    )
    if cleared is not None:
        data["cleared"] = rows_for(options.method, cleared, "summary-cleared")
    data["states"] = cell_state_rows(
        layout,
        "countrecall",
        [c.condition for c in data["present"]],
        native,
        contract=contract,
        seeds=options.seeds,
        sources=sources,
        accounting=options.state_accounting,
    )
    return data


def _matrix(per_task, seeds):
    """``[seeds, streams]`` of the mean per stream over the seeds present and the
    streams every present seed has (the strata are fixed by the task, so a
    stream missing for one seed is missing for all)."""
    present = [s for s in seeds if any(k[0] == s for k in per_task)]
    streams = sorted(
        {(k[1], k[2]) for k in per_task}
        & set.intersection(
            *[{(k[1], k[2]) for k in per_task if k[0] == s} for s in present]
        )
    )
    matrix = np.asarray(
        [[float(np.mean(per_task[(s, t, r)])) for t, r in streams] for s in present]
    )
    return matrix, present


def countrecall_streams(layout, contract, options, sources):
    """CountRecall continued over deck pairs (C3, appendix): exact accuracy per deck pair
    at the longest declared stream count every present cell has on every seed
    (the retained panels ``-s<N>``), with the joint seed/task interval and the
    per-seed values, the paired contrasts of the method against every other
    cell in every pair, and the method's stream-cleared companion (the carried
    state reset at every boundary) on the same pairs."""
    decisions = int(contract.environment.horizon)
    units = [
        (int(task), int(rollout))
        for task in contract.roster(options.split)
        for rollout in contract.evaluation.rollout_seeds
    ]
    rng = np.random.default_rng(options.resampling_seed)

    def cube(condition, streams, history, staged):
        layers = []
        for seed in options.seeds:
            events = load_streams_cell(
                layout,
                "countrecall",
                condition,
                options.split,
                history,
                streams,
                contract=contract,
                seeds=(seed,),
                sources=staged,
            )
            if events is None:
                return None
            layers.append(
                streams_matrix(
                    events, units=units, decisions=decisions, streams=streams
                )
            )
        return np.stack(layers)

    chosen, cubes = None, {}
    candidates = []
    for streams in options.streams:
        staged = []
        found = {
            c.condition: cube(c.condition, streams, "retained", staged)
            for c in options.cells
        }
        present = sum(v is not None for v in found.values())
        if present == len(options.cells):
            chosen, cubes = streams, found
            sources.extend(staged)
            break
        candidates.append((present, streams, found, staged))
    if chosen is None and candidates:
        present, streams, found, staged = max(candidates, key=lambda c: c[0])
        if present:
            chosen, cubes = streams, found
            sources.extend(staged)
    data = {
        "streams": chosen,
        "trained_pairs": C2_TRAINED_PAIRS,
        "present": [c for c in options.cells if cubes.get(c.condition) is not None],
        "missing": [
            c.condition for c in options.cells if cubes.get(c.condition) is None
        ],
        "rows": [],
        "contrasts": [],
        "cleared": [],
        "units": len(units),
    }
    if chosen is None:
        return data
    for condition, matrix in cubes.items():
        if matrix is None:
            continue
        for pair in range(chosen):
            data["rows"].append(
                {
                    "condition": condition,
                    "history": "retained",
                    "streams": chosen,
                    "pair": pair + 1,
                    **estimate(
                        matrix[:, :, pair],
                        samples=options.samples,
                        rng=rng,
                        seeds=options.seeds,
                    ),
                }
            )
    me = cubes.get(options.method)
    if me is not None:
        for condition, other in cubes.items():
            if other is None or condition == options.method:
                continue
            for pair in range(chosen):
                diff = me[:, :, pair] - other[:, :, pair]
                data["contrasts"].append(
                    {
                        "contrast": f"{rsm_name(options.method)} - {SHORT.get(condition, condition)}",
                        "left": options.method,
                        "right": condition,
                        "streams": chosen,
                        "pair": pair + 1,
                        **estimate(
                            diff, samples=options.samples, rng=rng, seeds=options.seeds
                        ),
                    }
                )
        staged = []
        cleared = cube(options.method, chosen, "attempt-cleared", staged)
        if cleared is not None:
            sources.extend(staged)
            for pair in range(chosen):
                data["cleared"].append(
                    {
                        "condition": options.method,
                        "history": "attempt-cleared",
                        "streams": chosen,
                        "pair": pair + 1,
                        **estimate(
                            cleared[:, :, pair],
                            samples=options.samples,
                            rng=rng,
                            seeds=options.seeds,
                        ),
                    }
                )
    return data


def mazerunner_size_rows(layout, contract, options, sources):
    """Goal fraction per cell, seed and maze size from the frozen endpoints
    (the adapter's ``-h<S>`` panels; at the trained size the plain endpoint
    panel stands in when the adapter's is absent)."""
    native = int(contract.environment.size)
    rows = []
    for cell in options.cells:
        for seed in options.seeds:
            for size in options.sizes:
                events = horizon_panel(
                    layout,
                    "mazerunner",
                    cell.condition,
                    seed,
                    options.split,
                    "retained",
                    size,
                    contract=contract,
                    sources=sources,
                )
                if events is None and size == native:
                    events = endpoint_panel(
                        layout,
                        "mazerunner",
                        cell.condition,
                        seed,
                        options.split,
                        "retained",
                        contract=contract,
                        sources=sources,
                    )
                if events is None:
                    continue
                result = maze_goal_fraction(events)
                rows.append(
                    {
                        "condition": cell.condition,
                        "seed": seed,
                        "size": size,
                        "budget": maze_budget(size, contract.environment),
                        "goal_fraction": result["goal_fraction"],
                        "full_sequence": result["full_sequence"],
                        "mean_decisions": result["mean_decisions"],
                        "goals_per_500_steps": result["goals_per_500_steps"],
                        "units": int(result["units"]),
                    }
                )
    return rows


def mazerunner_states(layout, contract, rows, options, sources):
    """Goal fraction and live state per cell at the largest maze every present
    cell has finished on every seed (``None`` when there is none)."""
    seen = sorted({r["condition"] for r in rows})
    complete = [
        size
        for size in sorted({r["size"] for r in rows})
        if all(
            {r["seed"] for r in rows if r["condition"] == c and r["size"] == size}
            >= set(options.seeds)
            for c in seen
        )
    ]
    if not complete:
        return {"size": None, "fraction": [], "states": []}
    size = complete[-1]
    rng = np.random.default_rng(options.resampling_seed)
    fraction = []
    for condition in seen:
        mine = [r for r in rows if r["condition"] == condition and r["size"] == size]
        per_seed = {r["seed"]: r["goal_fraction"] for r in mine}
        matrix = np.asarray([[per_seed[s]] for s in options.seeds])
        fraction.append(
            {
                "condition": condition,
                "size": size,
                **estimate(
                    matrix, samples=options.samples, rng=rng, seeds=options.seeds
                ),
            }
        )
    states = cell_state_rows(
        layout,
        "mazerunner",
        seen,
        size,
        contract=contract,
        seeds=options.seeds,
        sources=sources,
        accounting=options.state_accounting,
    )
    return {"size": size, "fraction": fraction, "states": states}


# ------------------------------------------------------------------- drawing


def _calls_axis(ax, horizon):
    step = 2000 if horizon >= 3000 else 500
    ticks = list(range(0, horizon + 1, step))
    ax.set_xlim(0, horizon)
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{t:,}" for t in ticks])
    ax.axvspan(WINDOW, horizon, color="#f4f4f4", zorder=0)
    ax.axvline(WINDOW, color=INK_FAINT, linewidth=0.7, linestyle=(0, (2, 2)), zorder=1)


def drawn_index(options, cell):
    """Where a cell sits in the drawing order, for a small offset that keeps
    overlapping range whiskers apart."""
    return [c.condition for c in options.cells].index(cell.condition)


def drawn(present, options):
    """The cells of ``options`` that have records in ``present`` (a prepared
    build holds every cell; each figure draws its own subset)."""
    conditions = {c.condition for c in present}
    return [c for c in options.cells if c.condition in conditions]


def missing_cells(data, options):
    """The cells of ``options`` without complete panels here, for the legend."""
    return [c for c in options.cells if c.condition in data["missing"]]


TITLE_ENVIRONMENT = {spec.title: key for key, spec in ENVIRONMENTS.items()}
"""Environment key by panel title, so a legend note follows its column."""


def _environment_of(title):
    return TITLE_ENVIRONMENT.get(title)


def drawn_cells_of(prepared, options):
    """The cells the figure's columns can show: every cell of ``options`` that
    has records in at least one drawn environment."""
    present = {c.condition for c in prepared["present"]}
    present |= {c.condition for c in prepared["countrecall"]["present"]}
    present |= set(prepared["mazerunner_seen"])
    return [c for c in options.cells if c.condition in present] or list(options.cells)


def column_title(options, environment, detail="", *, top=True):
    """The panel's title: the benchmark and its axis, the benchmark alone at
    the head of a compact column, or nothing under it."""
    name = ENVIRONMENTS[environment].title
    if options.compact:
        return name if top else ""
    return f"{name} · {detail}" if detail else name


KEYDOOR_PANEL_TITLES = {
    "rate": "(a) doors per 500-call window",
    "total": "(b) cumulative doors",
    "retention": "(c) retention against live state",
}
"""The main figure's panel titles: one benchmark, three views of it."""


def keydoor_title(data, options, view):
    if options.panel_titles == "short":
        return KEYDOOR_PANEL_TITLES[view]
    if data["horizon"] is None:
        return column_title(options, "keydoor", top=view != "retention")
    return column_title(
        options,
        "keydoor",
        f"{data['horizon']:,}-call task, {view}",
        top=view != "retention",
    )


def draw_doors_per_window(ax, data, options):
    """(a) completed doors per 500-call window against charged calls; the
    trained budget marked, the extension shaded; seeds as thin lines."""
    horizon = data["horizon"]
    top = 50.0
    if horizon:
        for cell in drawn(data["present"], options):
            mine = [r for r in data["windows"] if r["condition"] == cell.condition]
            x = [r["centre"] for r in mine]
            for seed in options.seeds:
                ax.plot(
                    x,
                    [r[f"seed_{seed}"] for r in mine],
                    color=cell.color,
                    linewidth=0.45,
                    alpha=0.35,
                    linestyle=cell.linestyle,
                    zorder=cell.zorder + 5,
                )
            band(ax, x, [r["lower"] for r in mine], [r["upper"] for r in mine], cell)
            ax.plot(x, [r["estimate"] for r in mine], **cell.markers(3.4))
            top = max(top, 1.05 * max(r["upper"] for r in mine))
        _calls_axis(ax, horizon)
        ax.text(
            WINDOW + 0.01 * horizon,
            0.97,
            "trained budget",
            transform=ax.get_xaxis_transform(),
            color=INK_MUTED,
            fontsize=8.0,
            va="top",
            ha="left",
        )
    ax.set_ylim(0, top)
    ax.set_xlabel("Charged calls")
    ax.set_ylabel(
        "Doors per 500-call window"
        if options.compact
        else "Completed doors per 500-call window"
    )
    ax.set_title(keydoor_title(data, options, "rate"))
    style_axes(ax)


def draw_cumulative_doors(ax, data, options):
    """(b) cumulative completed doors against charged calls."""
    horizon = data["horizon"]
    top = 10.0
    if horizon:
        for cell in drawn(data["present"], options):
            mine = [r for r in data["windows"] if r["condition"] == cell.condition]
            x = [0.0, *(r["last_call"] for r in mine)]
            band(
                ax,
                x,
                [0.0, *(r["cumulative_lower"] for r in mine)],
                [0.0, *(r["cumulative_upper"] for r in mine)],
                cell,
            )
            ax.plot(x, [0.0, *(r["cumulative_estimate"] for r in mine)], **cell.line())
            top = max(top, 1.05 * max(r["cumulative_upper"] for r in mine))
        _calls_axis(ax, horizon)
    ax.set_ylim(0, top)
    ax.set_xlabel("Charged calls")
    ax.set_ylabel("Cumulative completed doors")
    ax.set_title(keydoor_title(data, options, "total"))
    style_axes(ax)


def state_axis_label(options):
    """What the state axis counts, in the words of the chosen accounting."""
    return (
        "Peak live state per actor"
        if options.state_accounting == "peak"
        else "Average live state per actor"
    )


def _state_axis(ax, values):
    """Log x of live state per actor, ticks in KB and MB around the drawn values."""
    ax.set_xscale("log")
    low = min(values) if values else 1e4
    high = max(values) if values else 1e7
    ax.set_xlim(low / 4, high * 4)
    ticks = [t for t in (1e3, 1e4, 1e5, 1e6, 1e7, 1e8) if low / 4 <= t <= high * 4]
    if len(ticks) > 3:
        ticks = ticks[::2]
    labels = {
        1e3: "1 KB",
        1e4: "10 KB",
        1e5: "100 KB",
        1e6: "1 MB",
        1e7: "10 MB",
        1e8: "100 MB",
    }
    ax.set_xticks(ticks)
    ax.set_xticklabels([labels[t] for t in ticks])
    ax.minorticks_off()


ALLOCATION_TOLERANCE = 0.02
"""A cell's allocation is drawn beside its state only when the two differ by
more than this, so a fixed buffer is one marker and not two."""

ALLOCATION_NOTE = "hollow marker: allocated for the trained horizon"
"""What the second marker means, for the legend."""


def state_bytes_text(value):
    """Bytes as KB or MB, as a caption says them."""
    if value is None:
        return "-"
    return f"{value / 1e6:,.1f} MB" if value >= 1e6 else f"{value / 1e3:,.0f} KB"


def _quality_against_state(ax, cells, points, states, options, *, top, allocated=None):
    """One point per cell at its live state, no connecting frontier and no
    name: the figure's legend identifies the cells. The bar is the joint
    seed/task bootstrap interval, with the seeds as small markers behind it."""
    allocated = allocated or {}
    shown = [c for c in cells if c.condition in states and c.condition in points]
    drawn = [states[c.condition] for c in shown]
    drawn += [
        allocated[c.condition]
        for c in shown
        if c.condition in allocated
        and abs(allocated[c.condition] - states[c.condition])
        > ALLOCATION_TOLERANCE * max(allocated[c.condition], states[c.condition])
    ]
    for cell in shown:
        row, x = points[cell.condition], states[cell.condition]
        per_seed = [
            row[f"seed_{seed}"] for seed in options.seeds if f"seed_{seed}" in row
        ]
        low, high = row["lower"], row["upper"]
        for value in per_seed:
            ax.plot(
                x,
                value,
                marker=cell.marker,
                markersize=3.0,
                color=cell.color,
                alpha=0.45,
                linestyle="none",
                markerfacecolor=cell.marker_face,
                markeredgewidth=0.5,
                zorder=cell.zorder + 5,
            )
        ax.errorbar(
            x,
            row["estimate"],
            yerr=[[row["estimate"] - low], [high - row["estimate"]]],
            color=cell.color,
            elinewidth=0.7,
            capsize=0,
            marker=cell.marker,
            markersize=5.0,
            markerfacecolor=cell.marker_face,
            markeredgewidth=0.8,
            linestyle="none",
            zorder=cell.zorder + 10,
        )
        # What the cell reserved for the trained horizon, when that is not
        # what it holds here: a hollow marker joined to the filled one, so a
        # cell whose state happens to land on another's is still told apart.
        reserved = allocated.get(cell.condition)
        if reserved is not None and abs(reserved - x) > ALLOCATION_TOLERANCE * max(
            reserved, x
        ):
            ax.plot(
                [min(x, reserved), max(x, reserved)],
                [row["estimate"]] * 2,
                color=cell.color,
                linewidth=0.6,
                alpha=0.5,
                zorder=cell.zorder + 6,
            )
            ax.plot(
                reserved,
                row["estimate"],
                marker=cell.marker,
                markersize=4.4,
                color=cell.color,
                markerfacecolor="white",
                markeredgewidth=0.9,
                linestyle="none",
                zorder=cell.zorder + 9,
            )
        top = max(top, 1.05 * high)
    _state_axis(ax, drawn)
    ax.set_ylim(0, top)
    style_axes(ax, grid_axis="both")


def draw_retention_against_state(ax, data, options):
    """(d) Key-to-Door doors per 500 calls in the last window against the
    live inference state per actor averaged over the task."""
    horizon = data["horizon"]
    points, states, reserved = {}, {}, {}
    if horizon:
        last = max((r["window"] for r in data["windows"]), default=0)
        states = {r["condition"]: r["bytes"] for r in data["states"]}
        reserved = {r["condition"]: r["allocated_bytes"] for r in data["states"]}
        points = {r["condition"]: r for r in data["windows"] if r["window"] == last}
        first = (last - 1) * WINDOW + 1
        ax.set_ylabel(
            "Doors per 500 calls"
            if options.compact
            else f"Doors per 500 calls, calls {first:,} to {last * WINDOW:,}"
        )
        ax.set_xlabel(state_axis_label(options))
    else:
        ax.set_ylabel("Doors per 500 calls, last window")
        ax.set_xlabel(state_axis_label(options))
    _quality_against_state(
        ax,
        drawn(data["present"], options),
        points,
        states,
        options,
        top=50.0,
        allocated=reserved,
    )
    ax.set_title(keydoor_title(data, options, "retention"))


def _stratum_axis(ax, counts, *, compact=False):
    ax.set_xlim(-0.4, len(COUNT_RECALL_STRATA) - 0.6)
    ax.set_xticks(list(COUNT_RECALL_STRATA))
    ax.set_xticklabels(
        [
            ("in\nsegment" if s == 0 else f"{s}")
            + ("" if compact or s not in counts else f"\nn={counts[s]:,}")
            for s in COUNT_RECALL_STRATA
        ],
        fontsize=9.0,
    )
    ax.set_xlabel(
        "C32 rewrites since the evidence"
        if compact
        else "C32 segments since the queried suit was first dealt"
    )
    ax.set_ylim(0, 1.05)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])


def countrecall_title(data, options, view):
    if data["reference"] is None:
        return column_title(options, "countrecall", top=view != "state")
    return column_title(
        options, "countrecall", f"trained stream, {view}", top=view != "state"
    )


def draw_countrecall_retention(ax, data, options):
    """(b) exact accuracy by the segments since the first evidence, every cell
    on the same queries; the method's summary-cleared panel as the dotted
    trace; the opportunity count per stratum printed."""
    for cell in drawn(data["present"], options):
        mine = sorted(
            (r for r in data["rows"] if r["condition"] == cell.condition),
            key=lambda r: r["stratum"],
        )
        if not mine:
            continue
        x = [r["stratum"] for r in mine]
        band(ax, x, [r["lower"] for r in mine], [r["upper"] for r in mine], cell)
        ax.plot(x, [r["estimate"] for r in mine], **cell.markers(3.8))
    if data["cleared"] and options.show_cleared:
        mine = sorted(data["cleared"], key=lambda r: r["stratum"])
        ax.plot(
            [r["stratum"] for r in mine],
            [r["estimate"] for r in mine],
            **SUMMARY_CLEARED.markers(4.8),
        )
    counts = {
        r["stratum"]: r["queries_per_seed"]
        for r in data["rows"]
        if r["condition"] == data["reference"]
    }
    _stratum_axis(ax, counts, compact=options.compact)
    ax.set_ylabel(
        "Exact accuracy"
        if options.compact
        else "Exact accuracy on the stratum's queries"
    )
    ax.set_title(countrecall_title(data, options, "retention"))
    style_axes(ax)


def draw_countrecall_against_state(ax, data, options):
    """(e) exact accuracy on the queries whose evidence is the most rewrites
    old (the last stratum) against mean live state over the trained stream,
    so the row reads retained quality against the state it costs."""
    last = max((r["stratum"] for r in data["rows"]), default=None)
    points = {
        r["condition"]: r
        for r in data["rows"]
        if r["history"] == "retained" and r["stratum"] == last
    }
    states = {r["condition"]: r["bytes"] for r in data["states"]}
    reserved = {r["condition"]: r["allocated_bytes"] for r in data["states"]}
    ax.set_ylabel(
        "Exact accuracy"
        if options.compact
        else (
            f"Exact accuracy, evidence {last} rewrites old"
            if last
            else "Exact accuracy, evicted evidence"
        )
    )
    ax.set_xlabel(state_axis_label(options))
    _quality_against_state(
        ax,
        drawn(data["present"], options),
        points,
        states,
        options,
        top=1.05,
        allocated=reserved,
    )
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_title(countrecall_title(data, options, "state"))


def draw_countrecall_by_pair(ax, data, options):
    """CountRecall exact accuracy per continued deck pair (C3; the seed mean
    with its joint interval as a band), the trained lengths marked, and the
    method's stream-cleared companion as the cleared trace."""
    streams = data["countrecall_streams"]
    n = streams["streams"]
    if n:
        ax.axvspan(0.5, streams["trained_pairs"] + 0.5, color=GRID_FILL, zorder=0)
        ax.text(
            (streams["trained_pairs"] + 1) / 2,
            0.5,
            "trained lengths",
            fontsize=7.5,
            color=INK_FAINT,
            ha="center",
            va="center",
            rotation=90,
        )
    for cell in options.cells:
        mine = sorted(
            (r for r in streams["rows"] if r["condition"] == cell.condition),
            key=lambda r: r["pair"],
        )
        if not mine:
            continue
        if cell.condition in (options.method, ABLATION_CONDITION):
            band(
                ax,
                [r["pair"] for r in mine],
                [r["lower"] for r in mine],
                [r["upper"] for r in mine],
                cell,
            )
        ax.plot(
            [r["pair"] for r in mine],
            [r["estimate"] for r in mine],
            **cell.markers(3.2),
        )
    if streams["cleared"]:
        rows = sorted(streams["cleared"], key=lambda r: r["pair"])
        ax.plot(
            [r["pair"] for r in rows],
            [r["estimate"] for r in rows],
            **SUMMARY_CLEARED.line(),
        )
    if n:
        ax.set_xlim(0.5, n + 0.5)
        ticks = [1, 2, 4, 8, 16, 32] if n >= 8 else list(range(1, n + 1))
        ax.set_xticks([t for t in ticks if t <= n])
    ax.set_ylim(0, 1.05)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_xlabel("Deck pair in the continued task (104 records each)")
    ax.set_ylabel("Exact accuracy in the pair")
    title = ENVIRONMENTS["countrecall"].title + (
        f" · continued to {n} pairs (C3)" if n else ""
    )
    ax.set_title(title)
    style_axes(ax)


def draw_mazerunner_by_size(ax, data, options):
    """MazeRunner goal fraction against maze size under the area-scaled
    budget: seeds as faint markers, the seed mean as the line where every seed
    has a panel, the budget per size on the ticks."""
    rows = data["mazerunner"]
    sizes = list(options.sizes)
    budgets = data["mazerunner_budgets"]
    for cell in options.cells:
        mine = [r for r in rows if r["condition"] == cell.condition]
        if not mine:
            continue
        for seed in options.seeds:
            by_seed = sorted(
                (r for r in mine if r["seed"] == seed), key=lambda r: r["size"]
            )
            ax.plot(
                [r["size"] for r in by_seed],
                [r["goal_fraction"] for r in by_seed],
                marker=cell.marker,
                markersize=3.2,
                color=cell.color,
                alpha=0.4,
                linestyle="none",
                markerfacecolor=cell.marker_face,
                markeredgewidth=0.5,
                zorder=cell.zorder + 5,
            )
        complete = sorted(
            size
            for size in {r["size"] for r in mine}
            if {r["seed"] for r in mine if r["size"] == size} >= set(options.seeds)
        )
        means = [
            float(np.mean([r["goal_fraction"] for r in mine if r["size"] == size]))
            for size in complete
        ]
        ax.plot(complete, means, **cell.markers(3.8))
    ax.set_xlim(sizes[0] - 1, sizes[-1] + 1)
    ax.set_xticks(sizes)
    ax.set_xticklabels([str(s) for s in sizes])
    ax.set_ylim(0, 1.05)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_xlabel(
        "Maze size"
        if options.compact
        else f"Maze size (budget {budgets[sizes[0]]} to {budgets[sizes[-1]]:,} steps)"
    )
    ax.set_ylabel("Goal fraction (of 3 goals)")
    title = column_title(options, "mazerunner", "maze size" if rows else "")
    ax.set_title(title)
    style_axes(ax)


def draw_mazerunner_against_state(ax, data, options):
    """Goal fraction at the largest maze every cell has finished against mean
    live state over that task."""
    states_data = data["mazerunner_states"]
    size = states_data["size"]
    points = {r["condition"]: r for r in states_data["fraction"]}
    states = {r["condition"]: r["bytes"] for r in states_data["states"]}
    reserved = {r["condition"]: r["allocated_bytes"] for r in states_data["states"]}
    ax.set_ylabel(
        "Goal fraction"
        if options.compact or not size
        else f"Goal fraction at {size} x {size}"
    )
    ax.set_xlabel(state_axis_label(options))
    _quality_against_state(
        ax, options.cells, points, states, options, top=1.05, allocated=reserved
    )
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    title = column_title(
        options, "mazerunner", f"{size} x {size}, state" if size else "", top=False
    )
    ax.set_title(title)


GRID_FILL = "#f2f2f2"


def _wrap(text, width=118):
    """Break a caption line so it stays inside the page: at clause boundaries
    where they fall close enough to the width, and on words otherwise — a
    sentence without semicolons would else widen the whole figure."""
    out, line = [], ""
    for clause in text.split("; "):
        piece = clause if not line else line + "; " + clause
        if len(piece) > width and line:
            out.append(line + ";")
            line = clause
        else:
            line = piece
    out.append(line)
    return "\n".join(
        textwrap.fill(part, width=width, break_long_words=False) for part in out
    )


def state_coincidence(data, options):
    """When two cells' live state lands on the same spot of a state panel for
    a reason that does not hold at another task length, say so.

    On the CountRecall stream the lazy schedule peaks at a filled buffer one
    record before its next write, and at 104 records that peak happens to sit
    within a couple of per cent of the method's fixed buffer: the panel then
    shows no difference where there is one, so the note gives the reserved
    capacity and the same cell's peak on the longer Key-to-Door task."""
    rows = {r["condition"]: r for r in data["countrecall"]["states"]}
    me, other = rows.get(options.method), rows.get("memo")
    if me is None or other is None:
        return ""
    held, theirs = me["bytes"], other["bytes"]
    if abs(held - theirs) > 0.05 * max(held, theirs):
        return ""
    longer = {r["condition"]: r for r in data["states"]}.get("memo")
    grows = (
        f", and to {state_bytes_text(longer['bytes'])} on the "
        f"{data['horizon']:,}-call Key-to-Door task"
        if longer is not None and data["horizon"]
        else ""
    )
    return (
        f"On the {data['countrecall']['horizon'] + 1}-record stream Memo's "
        f"{state_bytes_text(theirs)} peak lands within "
        f"{100 * abs(held - theirs) / max(held, theirs):.0f} % of the method's "
        f"fixed {state_bytes_text(held)}: a coincidence of this task length, "
        "where the lazy schedule peaks one record before its next write. Memo "
        f"reserves {state_bytes_text(other['allocated_bytes'])} for the stream"
        f"{grows}, while the method holds the same state at every horizon."
    )


def caption(data, options):
    """The paired last-window readings, as the figure prints them."""
    if data["horizon"] is None:
        return ""
    me = rsm_name(options.method)
    against = {c.condition for c in options.cells}
    wanted = tuple(
        name
        for name, condition in (
            (f"H4b {me} - Memo", "memo"),
            (f"H4b {me} - full history", "full_context"),
            (f"{me} - GRU", "full_gru"),
            (f"{me} - Memo fixed", "memo_fixed"),
            (f"{me} - {rsm_name(ABLATION_CONDITION)}", ABLATION_CONDITION),
        )
        if condition in against
    )
    parts = []
    for row in data["contrasts"]:
        if row["contrast"] in wanted:
            name = row["contrast"].replace("H4b ", "")
            parts.append(
                f"{name} {row['estimate']:+.1f} [{row['lower']:+.1f}, {row['upper']:+.1f}]"
            )
    last = max(r["window"] for r in data["windows"])
    first = (last - 1) * WINDOW + 1
    head = (
        _wrap(
            f"Paired differences in calls {first:,} to {last * WINDOW:,} "
            f"({data['units']} tasks, {len(options.seeds)} seeds, joint seed/task 95 % "
            f"intervals): " + "; ".join(parts) + "."
        )
        if parts
        else f"Paired differences in calls {first:,} to {last * WINDOW:,}: pending "
        "until the method's horizon panels are written."
    )
    retention = data["countrecall"]
    pairs = ""
    if retention["rows"] and "countrecall" in options.environments:
        last = [r for r in retention["rows"] if r["stratum"] == COUNT_RECALL_STRATA[-1]]
        pairs = (
            "CountRecall accuracy on the queries whose evidence is three rewrites old: "
            + "; ".join(
                f"{SHORT.get(r['condition'], r['condition'])} {r['estimate']:.2f} [{r['lower']:.2f}, {r['upper']:.2f}]"
                for r in last
            )
            + "."
        )
    runner = ""
    if data["mazerunner"] and "mazerunner" in options.environments:
        largest = data["mazerunner_states"]["size"]
        if largest:
            runner = (
                f"MazeRunner goal fraction at {largest} x {largest} (every seed): "
                + "; ".join(
                    f"{SHORT.get(r['condition'], r['condition'])} {r['estimate']:.2f}"
                    for r in data["mazerunner_states"]["fraction"]
                )
                + "."
            )
    reading = (
        "the peak an actor holds within the task"
        if options.state_accounting == "peak"
        else "the mean an actor holds over the task's records"
    )
    other = "mean over the task" if options.state_accounting == "peak" else "peak"
    state = (
        f"State is {reading}, one accounting for every cell: Memo its recorded "
        "per-slot schedule; the full history one cache slot per record; the "
        f"others their fixed buffers; the {other} is in the saved table."
    )
    state += (
        " A hollow marker is what a cell allocated for its trained horizon, "
        "where that is not what it holds here."
    )
    coincidence = (
        state_coincidence(data, options)
        if "countrecall" in options.environments
        else ""
    )
    return "\n".join(
        _wrap(part) for part in (head, pairs, runner, state, coincidence) if part
    )


# --------------------------------------------------------------------- build


def prepare_figure3(layout, options=None):
    """Read every environment's frozen-endpoint panels once and compute what a
    figure draws (window rates and paired contrasts, CountRecall retention and
    continued pairs, MazeRunner sizes, live state), so the
    main Key-to-Door figure and the all-benchmark appendix figure are built
    from one read. Prepare with the widest cell set; each figure draws its own
    subset (:func:`drawn`)."""
    options = options or Figure3Options()
    sources = []
    computed = set(options.compute or PAPER_ENVIRONMENTS) | set(options.environments)
    keydoor_contract = layout.contract("keydoor")
    horizon, cubes = (
        choose_horizon(layout, keydoor_contract, options, sources)
        if "keydoor" in computed
        else (None, {c.condition: None for c in options.cells})
    )
    present = [c for c in options.cells if cubes.get(c.condition) is not None]
    missing = [c.condition for c in options.cells if cubes.get(c.condition) is None]
    windows = window_rates(
        cubes,
        samples=options.samples,
        seed=options.resampling_seed,
        seeds=options.seeds,
    )
    contrasts = paired_contrasts(
        cubes,
        samples=options.samples,
        seed=options.resampling_seed,
        seeds=options.seeds,
        method=options.method,
    )
    states = cell_states(layout, keydoor_contract, cubes, horizon, options, sources)
    checks = (
        [
            {
                "condition": c.condition,
                "seed": seed,
                "horizon": horizon,
                **prefix_status(layout, c.condition, seed, horizon, options.split),
            }
            for c in present
            for seed in options.seeds
        ]
        if horizon
        else []
    )
    countrecall_contract = layout.contract("countrecall")
    if "countrecall" in computed:
        countrecall = countrecall_retention(
            layout, countrecall_contract, options, sources
        )
        streams = countrecall_streams(layout, countrecall_contract, options, sources)
    else:
        countrecall = {
            "present": [],
            "missing": [c.condition for c in options.cells],
            "reference": None,
            "rows": [],
            "cleared": [],
            "endpoints": [],
            "states": [],
            "horizon": int(countrecall_contract.environment.horizon),
        }
        streams = {
            "streams": None,
            "trained_pairs": C2_TRAINED_PAIRS,
            "present": [],
            "missing": [c.condition for c in options.cells],
            "rows": [],
            "contrasts": [],
            "cleared": [],
            "units": 0,
        }
    mazerunner_contract = layout.contract("mazerunner")
    mazerunner_rows = (
        mazerunner_size_rows(layout, mazerunner_contract, options, sources)
        if "mazerunner" in computed
        else []
    )
    mazerunner_seen = {r["condition"] for r in mazerunner_rows}
    data = {
        "horizon": horizon,
        "present": present,
        "missing": missing,
        "units": int(cubes[present[0].condition].shape[1]) if present else 0,
        "windows": windows,
        "contrasts": contrasts,
        "states": states,
        "countrecall": countrecall,
        "countrecall_streams": streams,
        "mazerunner": mazerunner_rows,
        "mazerunner_budgets": {
            int(s): maze_budget(int(s), mazerunner_contract.environment)
            for s in options.sizes
        },
        "mazerunner_missing": {
            "missing": [
                c.condition for c in options.cells if c.condition not in mazerunner_seen
            ]
        },
        "mazerunner_states": mazerunner_states(
            layout, mazerunner_contract, mazerunner_rows, options, sources
        ),
    }
    return {
        "data": data,
        "countrecall": countrecall,
        "streams": streams,
        "cubes": cubes,
        "horizon": horizon,
        "present": present,
        "missing": missing,
        "windows": windows,
        "contrasts": contrasts,
        "states": states,
        "checks": checks,
        "mazerunner_rows": mazerunner_rows,
        "mazerunner_seen": mazerunner_seen,
        "sources": sources,
        "cells": list(options.cells),
    }


def build_figure3(layout, options=None, prepared=None):
    """One figure from prepared records (:func:`prepare_figure3`, called here
    when ``prepared`` is None): Dark Key-to-Door beyond the trained budget
    when ``environments`` is ``("keydoor",)``, otherwise one beyond-horizon
    axis per benchmark above a row of quality against live state."""
    options = options or Figure3Options()
    prepared = prepared if prepared is not None else prepare_figure3(layout, options)
    data = prepared["data"]
    countrecall, streams = prepared["countrecall"], prepared["streams"]
    horizon, present, missing = (
        prepared["horizon"],
        prepared["present"],
        prepared["missing"],
    )
    windows, contrasts, states = (
        prepared["windows"],
        prepared["contrasts"],
        prepared["states"],
    )
    checks = prepared["checks"]
    mazerunner_rows, mazerunner_seen = (
        prepared["mazerunner_rows"],
        prepared["mazerunner_seen"],
    )
    sources = list(prepared["sources"])
    columns_drawn = [e for e in PAPER_ENVIRONMENTS if e in options.environments]
    pending = {
        "Dark Key-to-Door": missing_cells(data, options),
        "CountRecall": missing_cells(countrecall, options),
        "MazeRunner": missing_cells(data["mazerunner_missing"], options),
    }
    handles = legend_handles(drawn_cells_of(prepared, options))
    if (
        countrecall["cleared"]
        and "countrecall" in columns_drawn
        and options.show_cleared
    ):
        # The summary-cleared trace of the retention panel.
        handles.append(SUMMARY_CLEARED.handle())
    handles.append(
        Line2D(
            [],
            [],
            label=ALLOCATION_NOTE,
            linestyle="none",
            marker="o",
            markersize=4.4,
            color=INK_MUTED,
            markerfacecolor="white",
            markeredgewidth=0.9,
        )
    )
    handles += pending_handles(
        {k: v for k, v in pending.items() if _environment_of(k) in columns_drawn},
        options.cells,
        reason="pending (no complete panels)",
    )
    columns = {
        "keydoor": (
            lambda ax: draw_doors_per_window(ax, data, options),
            lambda ax: draw_retention_against_state(ax, data, options),
        ),
        # The CountRecall test is retention within the trained stream: accuracy by the segments
        # since the evidence, on the single-pair cells; the continued pairs of C3
        # are a standalone appendix panel.
        "countrecall": (
            lambda ax: draw_countrecall_retention(ax, countrecall, options),
            lambda ax: draw_countrecall_against_state(ax, countrecall, options),
        ),
        "mazerunner": (
            lambda ax: draw_mazerunner_by_size(ax, data, options),
            lambda ax: draw_mazerunner_against_state(ax, data, options),
        ),
    }
    with paper_style():
        if columns_drawn == ["keydoor"]:
            # The paper's long-run figure: the rate beyond the trained budget,
            # the total over the task, and the last window against live state.
            fig = plt.figure(
                figsize=options.figsize or (FULL_WIDTH, 3.8), layout="constrained"
            )
            axes = fig.subplots(1, 3)
            draw_doors_per_window(axes[0], data, options)
            draw_cumulative_doors(axes[1], data, options)
            draw_retention_against_state(axes[2], data, options)

        elif len(columns_drawn) == 1:
            # One other environment: its beyond-horizon axis beside quality
            # against live state.
            fig = plt.figure(
                figsize=options.figsize or (FULL_WIDTH * 0.62, 3.5),
                layout="constrained",
            )
            axes = fig.subplots(1, 2)
            top, bottom = columns[columns_drawn[0]]
            top(axes[0])
            bottom(axes[1])
        else:
            # Row 1: the beyond-horizon axis, one column per benchmark.
            # Row 2: quality against live state per actor, the same columns.
            fig = plt.figure(
                figsize=options.figsize or (FULL_WIDTH, 7.0), layout="constrained"
            )
            axes = fig.subplots(2, len(columns_drawn))
            for column, environment in enumerate(columns_drawn):
                top, bottom = columns[environment]
                top(axes[0][column])
                bottom(axes[1][column])
        # The benchmark names the whole figure from the legend's title line:
        # a figure title and an outside legend would share the same margin.
        heading = (
            f"{ENVIRONMENTS['keydoor'].title} · one task of {horizon:,} charged "
            "calls, frozen policies"
            if options.panel_titles == "short" and horizon
            else None
        )
        composite_legend(
            fig,
            handles,
            ncol=3 if len(columns_drawn) == 1 else 4,
            title=heading,
        )
        text = caption(data, options) if options.caption else ""
        if text:
            fig.text(
                0.5, -0.01, text, ha="center", va="top", fontsize=8.5, color=INK_MUTED
            )
    size = options.panel_size

    def panel_handles(title, cells, extra=()):
        return (
            legend_handles(cells)
            + list(extra)
            + pending_handles(
                {title: pending[title]},
                options.cells,
                reason="pending (no complete panels)",
            )
        )

    keydoor_handles = panel_handles("Dark Key-to-Door", drawn(present, options))
    mazerunner_handles = panel_handles(
        "MazeRunner",
        [c for c in options.cells if c.condition in mazerunner_seen],
    )
    countrecall_handles = panel_handles(
        "CountRecall",
        drawn(countrecall["present"], options),
        [SUMMARY_CLEARED.handle()]
        if countrecall["cleared"] and options.show_cleared
        else [],
    )
    panels = {
        "keydoor_doors_per_window": standalone(
            lambda ax: draw_doors_per_window(ax, data, options),
            size=size,
            handles=keydoor_handles,
        ),
        "keydoor_cumulative_doors": standalone(
            lambda ax: draw_cumulative_doors(ax, data, options),
            size=size,
            handles=keydoor_handles,
        ),
        "keydoor_retention_against_state": standalone(
            lambda ax: draw_retention_against_state(ax, data, options),
            size=size,
            handles=keydoor_handles,
        ),
        "countrecall_accuracy_by_pair": standalone(
            lambda ax: draw_countrecall_by_pair(ax, data, options),
            size=size,
            handles=legend_handles(streams["present"])
            + ([SUMMARY_CLEARED.handle()] if streams["cleared"] else []),
        ),
        "mazerunner_goal_fraction_by_size": standalone(
            lambda ax: draw_mazerunner_by_size(ax, data, options),
            size=size,
            handles=mazerunner_handles,
        ),
        "mazerunner_goal_fraction_against_state": standalone(
            lambda ax: draw_mazerunner_against_state(ax, data, options),
            size=size,
            handles=mazerunner_handles,
        ),
        "countrecall_retention_by_segment": standalone(
            lambda ax: draw_countrecall_retention(ax, countrecall, options),
            size=size,
            handles=countrecall_handles,
        ),
        "countrecall_accuracy_against_state": standalone(
            lambda ax: draw_countrecall_against_state(ax, countrecall, options),
            size=size,
            handles=countrecall_handles,
        ),
    }
    tables = {
        "keydoor_windows": windows,
        "keydoor_contrasts": contrasts,
        "keydoor_state": states,
        "keydoor_prefix_checks": checks,
        "countrecall_retention": [*countrecall["rows"], *countrecall["cleared"]],
        "countrecall_endpoints": countrecall["endpoints"],
        "countrecall_state": countrecall["states"],
        "countrecall_streams": [*streams["rows"], *streams["cleared"]],
        "countrecall_stream_contrasts": streams["contrasts"],
        "mazerunner_sizes": mazerunner_rows,
        "mazerunner_state": [
            {
                **p,
                **{
                    k: v
                    for s in data["mazerunner_states"]["states"]
                    if s["condition"] == p["condition"]
                    for k, v in s.items()
                    if k not in ("condition", "horizon")
                },
            }
            for p in data["mazerunner_states"]["fraction"]
        ],
    }
    notes = {
        "keydoor": {
            "split": options.split,
            "horizon": horizon,
            "present": [c.condition for c in present],
            "missing": missing,
            "units": data["units"],
            "prefix_checks": sorted({c["status"] for c in checks}),
        },
        "countrecall": {
            "split": options.split,
            "present": [c.condition for c in countrecall["present"]],
            "missing": countrecall["missing"],
            "reference": countrecall["reference"],
            "strata": list(COUNT_RECALL_STRATA),
            "state": {r["condition"]: r["method"] for r in countrecall["states"]},
            "streams": streams["streams"],
            "streams_present": [c.condition for c in streams["present"]],
            "streams_missing": streams["missing"],
        },
        "mazerunner": {
            "split": options.split,
            "present": sorted(mazerunner_seen),
            "missing": data["mazerunner_missing"]["missing"],
            "sizes": sorted({r["size"] for r in mazerunner_rows}),
            "budgets": data["mazerunner_budgets"],
            "state_size": data["mazerunner_states"]["size"],
        },
        "seeds": list(options.seeds),
        "samples": options.samples,
        "resampling_seed": options.resampling_seed,
        "delta": DELTA,
        "checkpoint": "policy_epoch_999 (collection endpoint)",
        "interval": "95 % joint seed/task bootstrap, conditional on the trained seeds",
        "state": {r["condition"]: r["method"] for r in states},
    }
    panels = {
        key: figure
        for key, figure in panels.items()
        if any(key.startswith(environment) for environment in columns_drawn)
    }
    return BuiltFigure(options.name, fig, tables, notes, sources, panels)

In [ ]:
# Parameters: `horizons` are tried longest first; `cells` the memory carriers (no carry is left out of this figure);
# `method` the paper's RSM-O (the replacing write: its summary-cleared trace, the
# H4 readings and the CountRecall strata); `environments` the columns a figure
# draws; `samples` the bootstrap draw count (2000 in the paper).
#
# One read, two figures: `prepare_figure3` loads every benchmark's frozen
# horizon panels for the widest cell set (the appendix cells, Memo fixed
# included) and computes every estimate; the main Key-to-Door figure and the
# all-benchmark appendix figure are then drawn from that same payload, each
# with its own cells.
prepared = prepare_figure3(
    layout,
    Figure3Options(samples=2000, resampling_seed=0, cells=APPENDIX_HORIZON_CELLS),
)
print("prepared:", {k: v for k, v in prepared["data"]["mazerunner_missing"].items()})

In [ ]:
# Figure 3, the paper's long-run result: Dark Key-to-Door beyond the trained
# 500-call budget on the main text's carriers (the RSM family, the full
# history, the GRU, published Memo) — the rate per 500-call window, the
# cumulative doors and the last window against the live state it costs.
options = Figure3Options(
    samples=2000,
    resampling_seed=0,
    cells=HORIZON_CELLS,
    environments=("keydoor",),
    panel_titles="short",
)
figure3 = build_figure3(layout, options, prepared=prepared)
display(figure3.figure)

In [ ]:
# Appendix A5, one plot for every benchmark: the same axes on Key-to-Door,
# CountRecall (retention within the trained stream) and MazeRunner (larger
# mazes), with the quality-against-state row below and Memo fixed drawn.
appendix_options = Figure3Options(
    samples=2000,
    resampling_seed=0,
    cells=APPENDIX_HORIZON_CELLS,
    environments=PAPER_ENVIRONMENTS,
    panel_titles="compact",
    show_cleared=False,
    name="figureA5_beyond_the_training_horizon_all_benchmarks",
)
figure_a5 = build_figure3(layout, appendix_options, prepared=prepared)
display(figure_a5.figure)

In [ ]:
# Which horizon the Key-to-Door row sits on, which cells are pending, the
# CountRecall streams read and the MazeRunner sizes read.
print("keydoor    ", figure3.notes["keydoor"])
print("countrecall", figure3.notes["countrecall"])
print("mazerunner ", figure3.notes["mazerunner"])

In [ ]:
# The paired last-window differences, the CountRecall accuracy per continued
# pair and the MazeRunner goal fraction per size (every table is saved as CSV
# below).
display(Markdown(markdown_preview(figure3.tables["keydoor_contrasts"], limit=12)))
display(Markdown(markdown_preview(figure3.tables["countrecall_streams"], limit=32)))
display(Markdown(markdown_preview(figure3.tables["mazerunner_sizes"], limit=35)))

In [ ]:
# Save: each composite (PDF + PNG) and every panel (PNG) under FIGURES_DIR, one
# CSV per table and the notes under DATA_DIR.
for built in (figure3, figure_a5):
    for path in built.save(FIGURES_DIR, DATA_DIR):
        print(path.relative_to(REPO) if path.is_relative_to(REPO) else path)
    built.close()